# Caso 2 · Nodo Internet: predicción de bajas de clientes

**Equipo:** _completar nombres_

Este notebook sigue las consignas del enunciado (pasos 1 a 24). Cada paso tiene su celda; agreguen todas las que necesiten.
Las funciones de `src/` se completan en los archivos `.py`, no en el notebook. Antes de entregar: *Reiniciar y ejecutar todo*.

In [2]:
%load_ext autoreload
%autoreload 2
# autoreload vuelve a cargar los módulos de src/ cada vez que los modifican, sin reiniciar el notebook.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42

from src.preparacion import agregar_variables, normalizar_metodo_pago
from src.knn import EscaladorEstandar, KNNClasificador
from src.negocio import SUPUESTOS, ganancia_campania, lift, mejor_umbral, probabilidad_de_equilibrio, resultado_por_contacto

### Estado de las pruebas automáticas
Ejecuten esta celda cada vez que terminen una función de `src/`.

In [1]:
!python -m pytest -q --tb=no -rf

FFFFFFFFFFFFFFFFFFFFFF............FF                                     [100%]
=========================== short test summary info ===========================
FAILED tests/test_knn.py::test_escalador_igual_que_standard_scaler - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_escalador_columna_constante - NotImplementedError: Completar fit_transform
FAILED tests/test_knn.py::test_vecinos_igual_que_scikit_learn[euclidea] - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_vecinos_igual_que_scikit_learn[manhattan] - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_distancias_ordenadas_y_no_negativas - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_probabilidades_igual_que_scikit_learn[1-uniforme] - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_probabilidades_igual_que_scikit_learn[1-distancia] - NotImplementedError: Completar fit
FAILED tests/test_knn.py::test_probabilidades_igual_que_scikit_learn[5

## Parte A · Exploración y preparación (`src/preparacion.py`)

**Paso 1.** Carguen `datos/clientes_historico.csv` e informen filas, tipos, nulos por columna, cuentas de empleados, tasa de baja y valores distintos de `metodo_pago`.

In [3]:
# Tu código aquí
df = pd.read_csv('datos/clientes_historico.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_cliente              20000 non-null  str    
 1   fecha_corte             20000 non-null  str    
 2   segmento                20000 non-null  str    
 3   fecha_alta              20000 non-null  str    
 4   edad_titular            20000 non-null  int64  
 5   zona                    20000 non-null  str    
 6   plan_mbps               20000 non-null  int64  
 7   tipo_contrato           20000 non-null  str    
 8   fin_permanencia         8509 non-null   str    
 9   tiene_tv                20000 non-null  str    
 10  metodo_pago             20000 non-null  str    
 11  dia_vencimiento         20000 non-null  int64  
 12  cargo_mensual_gs        20000 non-null  int64  
 13  uso_mensual_gb          19411 non-null  float64
 14  dispositivos_promedio   20000 non-null  int64  
 

In [5]:
df.head()

,id_cliente,fecha_corte,segmento,fecha_alta,edad_titular,zona,plan_mbps,tipo_contrato,fin_permanencia,tiene_tv,...,uso_mensual_gb,dispositivos_promedio,velocidad_medida_pct,horas_sin_servicio_90d,reclamos_tecnicos_90d,visitas_tecnicas_12m,facturas_atrasadas_6m,consulto_baja_web_30d,motivo_baja,baja_90d
0,NI-118189,2026-05-31,residencial,2018-12-16,43,centro,300,12_meses,2026-12-16,no,...,242.1,3,88.0,0.7,0,1,0,0,NaN,0
1,NI-119690,2026-05-31,residencial,2023-12-03,33,este,600,mensual,NaN,no,...,327.1,3,99.2,0.7,0,0,0,0,competencia,1
2,NI-113950,2026-05-31,residencial,2017-10-23,22,sur,100,mensual,NaN,no,...,78.0,5,94.7,0.0,0,1,1,0,NaN,0
3,NI-118995,2026-05-31,residencial,2026-05-06,62,norte,300,12_meses,2027-05-06,no,...,587.1,3,NaN,3.7,1,0,0,0,NaN,0
4,NI-112506,2026-05-31,residencial,2020-05-20,47,oeste,300,12_meses,2027-05-21,no,...,823.2,10,NaN,0.0,0,0,1,0,NaN,0


### Total de empleados

In [6]:
df['segmento'].unique()

<StringArray>
['residencial', 'empleado']
Length: 2, dtype: str

In [7]:
total_empleados = (df['segmento'] == 'empleado').sum()
total_empleados

np.int64(612)

hay un total de 612 empleados

### Tasa de baja

In [8]:
tasa_bajas = df['baja_90d'].mean()
tasa_bajas

np.float64(0.07675)

La tasa de bajas es del 7.68%

### Valores distintos que figuran en metodo_pago

In [9]:
df['metodo_pago'].unique()

<StringArray>
[     'debito_automatico',          'transferencia',          'Transferencia',
               'efectivo',      'Debito_Automatico',        'tarjeta_credito',
 'transferencia bancaria',              'efectivo ',        'TARJETA CREDITO',
        'Tarjeta_Credito',      'DEBITO AUTOMATICO',               'EFECTIVO',
     'Tarjeta de crédito',          'TRANSFERENCIA',      'debito automatico',
               'Efectivo',        'tarjeta credito',      'Débito automático']
Length: 18, dtype: str

**Paso 2.** Implementen `normalizar_metodo_pago` y `agregar_variables` hasta que pasen `tests/test_preparacion.py`. Apliquen la función y excluyan las cuentas de empleados.

In [10]:
# Tu código aquí
df_clean = agregar_variables(df)

**Paso 3.** Tasa de baja por tipo de contrato, `sin_permanencia_3m`, zona, antigüedad, reclamos técnicos y consulta de baja web; zona este con cargo mayor a Gs. 280.000 contra el resto. Tres hallazgos para el negocio.

In [ ]:

tasa_contrato = (
    df_clean.groupby('tipo_contrato')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

tasa_permanencia = (
    df_clean.groupby('sin_permanencia_3m')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

tasa_zona = (
    df_clean.groupby('zona')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

tasa_reclamos = (
    df_clean.groupby('reclamos_tecnicos_90d')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

tasa_web = (
    df_clean.groupby('consulto_baja_web_30d')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

In [12]:
df_clean['antiguedad_meses'].unique()

array([ 89.46201232,  29.8973306 , 103.22792608, ...,  82.49691992,
        70.89938398,  90.25051335], shape=(3277,))

In [ ]:

df_clean['tramo_antiguedad'] = pd.cut(
    df_clean['antiguedad_meses'],
    bins=[-1, 6, 12, 24, 48, np.inf],
    labels=[
        '0-6 meses',
        '6-12 meses',
        '12-24 meses',
        '24-48 meses',
        'Más de 48m',
    ],
)

tasa_antiguedad = (
    df_clean.groupby('tramo_antiguedad', observed=False)['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)



condicion_este_alto = (df_clean['zona'].str.lower() == 'este') & (
    df_clean['cargo_mensual_gs'] > 280000
)

df_clean['grupo_segmento'] = np.where(
    condicion_este_alto, 'Zona Este (>280k Gs)', 'Resto de Clientes'
)

tasa_segmento_especifico = (
    df_clean.groupby('grupo_segmento')['baja_90d']
    .agg(total_clientes='count', tasa_baja_pct=lambda x: x.mean() * 100)
    .round(2)
)

In [14]:
print('=== TASA DE BAJA POR TIPO DE CONTRATO ===\n', tasa_contrato, '\n')
print(
    '=== TASA DE BAJA POR SIN PERMANENCIA 3M ===\n', tasa_permanencia, '\n'
)
print('=== TASA DE BAJA POR ZONA ===\n', tasa_zona, '\n')
print('=== TASA DE BAJA POR ANTIGÜEDAD ===\n', tasa_antiguedad, '\n')
print('=== TASA DE BAJA POR RECLAMOS TÉCNICOS ===\n', tasa_reclamos, '\n')
print('=== TASA DE BAJA POR CONSULTA WEB BAJA ===\n', tasa_web, '\n')
print('=== SEGMENTO ZONA ESTE (>280K) VS RESTO ===\n', tasa_segmento_especifico)

=== TASA DE BAJA POR TIPO DE CONTRATO ===
                total_clientes  tasa_baja_pct
tipo_contrato                               
12_meses                 5753           5.28
24_meses                 2756           4.03
mensual                 11491           9.75 

=== TASA DE BAJA POR SIN PERMANENCIA 3M ===
                     total_clientes  tasa_baja_pct
sin_permanencia_3m                               
0                             7366           4.86
1                            12634           9.32 

=== TASA DE BAJA POR ZONA ===
         total_clientes  tasa_baja_pct
zona                                 
centro            3637           5.50
este              4051          14.93
norte             4381           5.46
oeste             3570           7.17
sur               4361           5.39 

=== TASA DE BAJA POR ANTIGÜEDAD ===
                   total_clientes  tasa_baja_pct
tramo_antiguedad                               
0-6 meses                   3294           8.74
6-1

**Paso 4.** Columnas que no pueden usarse como variables y por qué. Demuestren con una tabla el problema de la columna que haría parecer excelente a cualquier modelo.

In [ ]:

tiene_motivo = df_clean['motivo_baja'].notna().map(
    {True: 'Tiene Motivo (Not NaN)', False: 'Sin Motivo (NaN)'}
)


tabla_leakage = pd.crosstab(
    tiene_motivo,
    df_clean['baja_90d'].map({1: 'Baja (1)', 0: 'Activo (0)'}),
    margins=True,
    margins_name='Total',
)

print(tabla_leakage)

baja_90d                Activo (0)  Baja (1)  Total
motivo_baja                                        
Sin Motivo (NaN)             18465         0  18465
Tiene Motivo (Not NaN)           0      1535   1535
Total                        18465      1535  20000


**Paso 5.** División 75/25 estratificada por `baja_90d` con `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

columnas_a_excluir = [
    'baja_90d',
    'motivo_baja',
    'id_cliente',
    'fecha_corte',
    'fecha_alta',
    'fin_permanencia',
]

X = df_clean.drop(columns=columnas_a_excluir)
y = df_clean['baja_90d']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## Parte B · Preprocesamiento y KNN desde cero (`src/knn.py`)

**Paso 6.** `ColumnTransformer`: imputación por mediana, `np.log1p` en `uso_mensual_gb` y `horas_sin_servicio_90d`, estandarización y `OneHotEncoder(handle_unknown="ignore")`.

In [ ]:
# Tu código aquí

**Paso 7.** Implementen `EscaladorEstandar` y `KNNClasificador` hasta que pasen `tests/test_knn.py`.

In [ ]:
# Tu código aquí

**Paso 8.** Su `KNNClasificador` contra `KNeighborsClassifier` con k=51 y pesos por distancia: probabilidades iguales y tiempo de cada uno.

In [ ]:
# Tu código aquí

## Parte C · Ajuste de KNN

**Paso 9.** AP de KNN con k=25, con y sin estandarizar (validación cruzada estratificada de 5 particiones). Expliquen la diferencia.

In [ ]:
# Tu código aquí

**Paso 10.** `GridSearchCV` con `average_precision`: k, pesos y `p`. Gráfico de AP en función de k y explicación del k elegido.

In [ ]:
# Tu código aquí

**Paso 11.** AP sin las variables de permanencia y, por separado, sin `edad_titular`, `dispositivos_promedio` ni `dia_vencimiento`. Conclusión.

In [ ]:
# Tu código aquí

## Parte D · Naive Bayes gaussiano como comparación

**Paso 12.** AP de `GaussianNB` con el mismo preprocesamiento, con y sin transformación logarítmica.

In [ ]:
# Tu código aquí

**Paso 13.** Correlación de Spearman entre las variables de calidad del servicio y relación con el supuesto de independencia.

In [ ]:
# Tu código aquí

**Paso 14.** Probabilidades fuera de muestra de los dos modelos: puntaje de Brier e histogramas. ¿Cuál exagera más su confianza?

In [ ]:
# Tu código aquí

## Parte E · Economía de la campaña (`src/negocio.py`)

**Paso 15.** Implementen las funciones de `src/negocio.py` hasta que pasen `tests/test_negocio.py`. Informen G⁺, G⁻ y la probabilidad de equilibrio.

In [ ]:
# Tu código aquí

**Paso 16.** Regla comercial sobre el conjunto de entrenamiento: clientes marcados, proporción que se fue y ganancia por cada 1.000 clientes.

In [ ]:
# Tu código aquí

**Paso 17.** Umbral de cada modelo con `mejor_umbral` sobre probabilidades fuera de muestra, con un máximo del 7% de los clientes. Comparación con la probabilidad de equilibrio.

In [ ]:
# Tu código aquí

**Paso 18.** Evaluación única en prueba: tabla de estrategias y curva de ganancia acumulada.

In [ ]:
# Tu código aquí

## Parte F · Explicación para Fidelización

**Paso 19.** Importancia por permutación del modelo elegido sobre prueba (`average_precision`).

In [ ]:
# Tu código aquí

**Paso 20.** Función `explicar_cliente(id_cliente)` y su aplicación a los 3 clientes de mayor riesgo de la campaña, con la oferta propuesta para cada uno.

In [ ]:
# Tu código aquí

## Parte G · Lista de la campaña de octubre

**Paso 21.** Reentrenen el modelo con todos los clientes residenciales del histórico y calculen la probabilidad de baja de `datos/clientes_campania.csv`.

In [ ]:
# Tu código aquí

**Paso 22.** Generen `campania_octubre.csv` (6.000 filas; columnas `id_cliente`, `prob_baja`, `contactar`) y verifiquen: como máximo 420 contactos, ningún empleado, ningún cliente repetido.

In [ ]:
# Tu código aquí

**Paso 23.** Ganancia esperada de la campaña con la precisión observada en prueba.

In [ ]:
# Tu código aquí

## Parte H · Informe ejecutivo

**Paso 24.** El informe para Martín Aquino se entrega aparte, en `informe_ejecutivo.pdf`.

## Preguntas de reflexión profesional

Respondan cada pregunta en un párrafo, apoyándose en sus resultados.

**1. `motivo_baja` no puede ser variable del modelo, pero sí se usó para explicar a los vecinos. ¿Por qué un uso es incorrecto y el otro no?**

_Respuesta:_ 

**2. ¿Por qué KNN necesita estandarizar las variables y Naive Bayes gaussiano no?**

_Respuesta:_ 

**3. ¿Qué dice la diferencia entre el umbral elegido y la probabilidad de equilibrio sobre las probabilidades de KNN? ¿Qué le responderían a Finanzas?**

_Respuesta:_ 

**4. ¿Cómo medirían el efecto real de la campaña de octubre?**

_Respuesta:_ 

**5. ¿Es aceptable ofrecer descuentos solo a algunos clientes según su zona o su perfil? ¿Qué riesgos ven?**

_Respuesta:_ 

**6. ¿Qué implica que KNN guarde los datos de entrenamiento para actualizar el modelo y para la privacidad?**

_Respuesta:_ 

**7. ¿Qué pudo cambiar entre la foto de mayo y la de septiembre y cómo lo detectarían antes de lanzar la campaña?**

_Respuesta:_ 